# Planning-horizon sweep (Fig. 9)

Goal-conditioned planning success vs. goal offset (evaluation budget = 2 x offset),
one panel per environment. The figure is displayed inline.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def _repo_root(start):
    for c in [start.resolve(), *start.resolve().parents]:
        if (c / "pyproject.toml").exists() and (c / "experiments").is_dir():
            return c
    return start.resolve()


REPO_ROOT = _repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from plot_style import FULL_WIDTH, apply_matplotlib_style, palette

apply_matplotlib_style()

CSV_PATH = REPO_ROOT / "experiments" / "horizon_sweep" / "aggregated_results.csv"

ENV_ORDER = ["TwoRoom", "Reacher", "Push-T", "OGBench-Cube"]
METHOD_ORDER = ["Our", "SIGReg", "Random", "Forward only"]
HORIZONS = [25, 40, 55, 70, 85, 100]

METHOD_COLORS = {
    "Our": palette["Dark Red"],
    "SIGReg": palette["Dark Blue"],
    "Random": palette["Dark Grey"],
    "Forward only": palette["Dark Purple"],
}
METHOD_STYLES = {
    "Our": {"marker": "D", "linestyle": "-"},
    "SIGReg": {"marker": "D", "linestyle": "-"},
    "Random": {"marker": "D", "linestyle": "--"},
    "Forward only": {"marker": "D", "linestyle": "-."},
}
print(CSV_PATH)

In [ ]:
ENV_LABELS = {"tworoom": "TwoRoom", "reacher": "Reacher", "pusht": "Push-T", "ogbcube": "OGBench-Cube"}
METHOD_LABELS = {"inverse": "Our", "sigreg": "SIGReg", "random": "Random", "forward_only": "Forward only"}

df = pd.read_csv(CSV_PATH)
if "status" in df.columns:
    df = df[df["status"].eq("ok")].copy()
df["success_rate"] = pd.to_numeric(df["success_rate"], errors="coerce")
df["goal_offset"] = pd.to_numeric(df["goal_offset"], errors="coerce")
df["environment"] = df["env"].map(ENV_LABELS)
df["method_label"] = df["method"].map(METHOD_LABELS)
df = df.dropna(subset=["success_rate", "goal_offset", "environment", "method_label"])
df["goal_offset"] = df["goal_offset"].astype(int)
if df.empty:
    raise ValueError("No completed horizon-sweep rows. Run aggregate_results.py after jobs finish.")

mean = df.groupby(["environment", "method_label", "goal_offset"])["success_rate"].mean()
success_by_horizon = {
    env: {m: [float(mean.get((env, m, h), np.nan)) for h in HORIZONS] for m in METHOD_ORDER}
    for env in ENV_ORDER
}
display(mean.unstack("goal_offset"))

In [ ]:
def plot_env_horizon_sweep(ax, env_name):
    for method in METHOD_ORDER:
        ax.plot(
            HORIZONS,
            success_by_horizon[env_name][method],
            color=METHOD_COLORS[method],
            marker=METHOD_STYLES[method]["marker"],
            linestyle=METHOD_STYLES[method]["linestyle"],
            linewidth=1.25,
            markersize=3.6,
            markerfacecolor="white",
            markeredgewidth=0.8,
            label=method,
        )
    ax.set_title(env_name, pad=3)
    ax.set_xlim(20, 105)
    ax.set_ylim(0, 100)
    ax.set_xticks(HORIZONS)
    ax.set_yticks([0, 50, 100])
    ax.grid(axis="y", linewidth=0.45)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


fig, axes = plt.subplots(1, len(ENV_ORDER), figsize=(FULL_WIDTH, 1.68), sharey=True)
for ax, env_name in zip(axes, ENV_ORDER):
    plot_env_horizon_sweep(ax, env_name)

axes[0].set_ylabel("Success rate (%)")
for ax in axes[1:]:
    ax.tick_params(axis="y", length=0)
for ax in axes:
    ax.set_xlabel("Goal offset")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels, loc="upper center", ncol=len(METHOD_ORDER),
    bbox_to_anchor=(0.56, 1.02), columnspacing=0.95, handlelength=1.35,
)
fig.subplots_adjust(left=0.075, right=0.995, bottom=0.27, top=0.78, wspace=0.16)
plt.show()